In [ ]:
import sys
import os
from pathlib import Path
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Hardware Acceleration: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("Running in CPU mode. For training, a T4/V100/A100 GPU is recommended.")

try:
    import ultralytics
    print(f"Ultralytics Version: {ultralytics.__version__}")
except ImportError:
    print("Installing Ultralytics...")
    !pip install ultralytics pandas matplotlib seaborn opencv-python --quiet
    import ultralytics
    print(f"Ultralytics Installed: {ultralytics.__version__}")

In [ ]:
import yaml
data_yaml_content = {'path': '../dataset', 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'rock'}}
data_yaml_path = Path('rock_data.yaml')
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)
print(f'Created dataset configuration file: {data_yaml_path.resolve()}')
print(yaml.dump(data_yaml_content, default_flow_style=False))

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-seg.pt')
training_params = {'data': str(data_yaml_path), 'epochs': 500, 'imgsz': 640, 'batch': 32, 'workers': 8, 'optimizer': 'SGD', 'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'close_mosaic': 10, 'save': True, 'project': 'runs/segment', 'name': 'boulder_edge_seg', 'exist_ok': True}
print('Training Pipeline Configuration:')
for k, v in training_params.items():
    print(f'  {k}: {v}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except ImportError:
    plt.style.use('default')
csv_candidates = [Path('../BOULDER/results.csv'), Path('BOULDER/results.csv'), Path('../runs/segment/boulder_edge_seg/results.csv')]
results_file = next((p for p in csv_candidates if p.exists()), None)
if results_file:
    print(f'Loading empirical training logs from: {results_file.resolve()}')
    df = pd.read_csv(results_file)
    df.columns = df.columns.str.strip()
    print(f'Total Epochs Completed: {len(df)}')
    print(f'Columns Recorded: {list(df.columns)}')
    final_epoch = df.iloc[-1]
    print('\n=== Final Converged Validation Metrics ===')
    print(f"Box Precision:  {final_epoch.get('metrics/precision(B)', 0):.4f}")
    print(f"Box Recall:     {final_epoch.get('metrics/recall(B)', 0):.4f}")
    print(f"Box mAP@50:     {final_epoch.get('metrics/mAP50(B)', 0):.4f}")
    print(f"Box mAP@50-95:  {final_epoch.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"Mask Precision: {final_epoch.get('metrics/precision(M)', 0):.4f}")
    print(f"Mask Recall:    {final_epoch.get('metrics/recall(M)', 0):.4f}")
    print(f"Mask mAP@50:    {final_epoch.get('metrics/mAP50(M)', 0):.4f}")
    print(f"Mask mAP@50-95: {final_epoch.get('metrics/mAP50-95(M)', 0):.4f}")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    sns.set_theme(style='whitegrid')
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='#1f77b4', lw=1.8)
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='#ff7f0e', lw=1.8, linestyle='--')
    axes[0, 0].set_title('Bounding Box Regression Loss (CIoU)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 1].plot(df['epoch'], df['train/seg_loss'], label='Train Mask Seg Loss', color='#2ca02c', lw=1.8)
    axes[0, 1].plot(df['epoch'], df['val/seg_loss'], label='Val Mask Seg Loss', color='#d62728', lw=1.8, linestyle='--')
    axes[0, 1].set_title('Instance Mask Loss (Proto-BCE)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 2].plot(df['epoch'], df['train/dfl_loss'], label='Train DFL Loss', color='#9467bd', lw=1.8)
    axes[0, 2].plot(df['epoch'], df['val/dfl_loss'], label='Val DFL Loss', color='#8c564b', lw=1.8, linestyle='--')
    axes[0, 2].set_title('Distribution Focal Loss (DFL)', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].legend()
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='Box mAP@50', color='#007acc', lw=2)
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Box mAP@50-95', color='#ff6600', lw=2)
    axes[1, 0].set_title('Bounding Box mAP Convergence', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 1].plot(df['epoch'], df['metrics/mAP50(M)'], label='Mask mAP@50', color='#2ca02c', lw=2)
    axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(M)'], label='Mask mAP@50-95', color='#d62728', lw=2)
    axes[1, 1].set_title('Mask Segmentation mAP Convergence', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 2].plot(df['epoch'], df['metrics/precision(M)'], label='Mask Precision', color='#17becf', lw=1.8)
    axes[1, 2].plot(df['epoch'], df['metrics/recall(M)'], label='Mask Recall', color='#bcbd22', lw=1.8)
    axes[1, 2].set_title('Mask Precision & Recall Trajectory', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].legend()
    plt.tight_layout()
    plt.show()
else:
    print('Results CSV not found. Please ensure BOULDER/results.csv is in the repository.')

In [ ]:
from PIL import Image
val_labels_path = Path('../BOULDER/val_batch1_labels.jpg')
val_preds_path = Path('../BOULDER/val_batch1_pred.jpg')
if not val_labels_path.exists():
    val_labels_path = Path('BOULDER/val_batch1_labels.jpg')
    val_preds_path = Path('BOULDER/val_batch1_pred.jpg')
if val_labels_path.exists() and val_preds_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    img_labels = Image.open(val_labels_path)
    axes[0].imshow(img_labels)
    axes[0].set_title('Ground Truth Validation Batch Annotations', fontsize=13, fontweight='bold')
    axes[0].axis('off')
    img_preds = Image.open(val_preds_path)
    axes[1].imshow(img_preds)
    axes[1].set_title('Trained Model Predictions (Box + Segmentation Mask)', fontsize=13, fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Validation sample images not found in BOULDER directory.')

In [ ]:
import numpy as np
import cv2

def evaluate_boulder_hazard(mask, frame_shape):
    height, width = frame_shape[:2]
    total_area = height * width
    mask_pixels = np.count_nonzero(mask)
    area_ratio = mask_pixels / total_area
    corridor_x_min = int(width * 0.3)
    corridor_x_max = int(width * 0.7)
    central_slice = mask[:, corridor_x_min:corridor_x_max]
    central_overlap = np.any(central_slice > 0)
    bottom_slice = mask[int(height * 0.9):, :]
    touches_bottom = np.any(bottom_slice > 0)
    if area_ratio > 0.3 and central_overlap:
        severity = 'critical'
    elif area_ratio > 0.15 or (central_overlap and touches_bottom):
        severity = 'medium'
    else:
        severity = 'low'
    return {'mask_area_ratio': area_ratio, 'central_corridor_overlap': central_overlap, 'touches_bottom': touches_bottom, 'severity': severity}
test_mask = np.zeros((320, 320), dtype=np.uint8)
cv2.circle(test_mask, (160, 200), 75, 255, -1)
hazard_metrics = evaluate_boulder_hazard(test_mask, (320, 320))
print('Corridor Blockage Assessment Test:')
for k, v in hazard_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
model_path = Path('../models/hazards/boulder.pt')
if not model_path.exists():
    model_path = Path('models/hazards/boulder.pt')
if model_path.exists():
    print(f'Exporting model from: {model_path.resolve()}')
    edge_model = YOLO(str(model_path))
    try:
        onnx_file = edge_model.export(format='onnx', imgsz=320, half=False, simplify=True)
        print(f'Successfully exported to ONNX: {onnx_file}')
    except Exception as e:
        print(f'ONNX export preview: {e}')
else:
    print(f'Checkpoint not found at {model_path}. Provide trained weights to export.')